# HW1: Building a Gymnasium Environment for Underwater Caldera Exploration

Our setting is an underwater caldera exploration mission, in which an autonomous vehicle navigates through a volcanic basin and collects scientific measurements from different locations.

The vehicle operates under realistic physical and operational constraints. It has limited fuel, which restricts how far it can travel. In addition, when the vehicle moves on the ocean surface, its motion may be influenced by water currents, introducing uncertainty into its movement.

In this assignment, we will model this mission as a simulated environment by specifying:

the state of the system,
the actions available to the vehicle,
the dynamics governing movement and uncertainty,
and the reward function that encodes the mission setting.

This simulation provides a structured representation of the environment, enabling systematic interaction and analysis of the system’s behavior under different conditions. It will allow us to study and design algorithms for planning and decision-making in challenging exploration settings.

<img src="../images/Kolumbo-submarine-volcano-and-caldera-Alexandri-et-al.jpg" alt="Kolumbo submarine caldera" width="500" />


When considering autonomous agents, we need to consider the environment with which they interact. They choose actions to perform and recieve from the environment observations and rewards. In this assignment, we focus on the **environment**, and on modeling the exploration task. 

<img src="../images/agent-env.png" alt="agent-env" width="500" />


## What is Gymnasium and why are we using it?



Your text is already very strong—here is a slightly polished, tighter version that improves flow and removes small redundancies while keeping your intent intact:

---

**Gymnasium** is a standard library for building reinforcement learning environments. It provides a clear structure for describing how an agent interacts with an environment over time.

[https://gymnasium.farama.org/index.html](https://gymnasium.farama.org/index.html)

A Gymnasium environment defines:

* an **observation space** describing what the agent can observe
* an **action space** describing what the agent can do
* a `reset()` method for starting a new episode
* a `step(action)` method for applying an action and returning the result

Each call to `step()` returns:

```python
observation, reward, terminated, truncated, info
```

In this interface, the environment is responsible for returning correct and consistent values.

 The `reset()` function initializes a new episode and returns an initial observation `obs` (which must belong to the `observation_space`) along with an optional `info` dictionary. 
 
 The `step(action)` function applies an action (which must belong to the `action_space`) and returns a tuple `(obs, reward, terminated, truncated, info)`, where `obs` is the next observation, `reward` is a scalar signal reflecting the immediate outcome of the action, `terminated` indicates whether the episode ended due to the task’s internal logic (e.g., goal reached or failure), and `truncated` indicates whether the episode was stopped due to an external constraint such as a time limit. The `info` dictionary can be used for additional diagnostics.

Together, these components define the transition dynamics and reward function of the underlying decision process, meaning that the environment effectively implements the MDP (or POMDP) that the agent is learning to solve.



In this assignment, we are building a custom **Gymnasium environment** for an underwater caldera exploration mission. We are using Gymnasium here because it gives our underwater exploration problem a clean, reusable RL interface that will work well for simulation, debugging, training, and evaluation.



That means we need to define:

- what the agent observes about the mission state
- what actions it can take while exploring
- what reward it receives for performing actions
- when an episode should end



## Mission story


The robotic agent in our setting operates on the water surface. The environment is therefore modeled as a 2D grid laid over an underwater caldera.

At each time step, the agent can:

- move north
- move south
- move east
- move west
- take a sample at its current location

Depending on the task, some grid cells may be more scientifically valuable than others. For example, the agent might aim to detect specific chemical signatures or anomalies, in which case chemically unusual regions would yield higher rewards when sampled. In another scenario, the agent may seek to identify particular features of the environment, such as the point of maximal depth, where the reward reflects the informational value of each measurement.

These different objectives can be naturally expressed through different formulations of the reward function, allowing the same environment to support a variety of scientific goals. This is known as **reward shaping**, the practice of designing or augmenting the reward signal to guide the agent toward desired behaviors while preserving the underlying task objectives.

### Modeling the Exploration Task


We design the exploration environments in terms of two separate layers: one for representing the environment, and the second for representing the exploration task.


First, we define a continuous field that represents the map environment and its underlying phenomenon as a function
$
f : \mathbb{R}^2 \rightarrow \mathbb{R}.
$
As is common in spatial modeling, we use a {\bf mixture of Gaussians} to capture structured variability across space:
$$
f(x,y) = \sum_{i=1}^{K} w_i \exp\left(

\frac{(x - \mu_{x,i})^2}{2\sigma_{x,i}^2} \frac{(y - \mu_{y,i})^2}{2\sigma_{y,i}^2}
  \right)
  $$
  This field can represent quantities such as depth, chemical concentration, anomaly likelihood, or information value, and is defined independently of any particular grid or discretization.

It is withint this abstraction that we represent the agent's movements and observations. 

Below is an visualization of an example environment, with the agent marked by a black dot. 


<img src="../images/caldera-map.png" alt="agent-env" width="500" />


The second layer is an abstraction introduced to support the exploration task. In this discretization layer, the grid world is obtained by sampling the continuous field at discrete locations; that is, each cell 
$(x,y)$ corresponds to a point at which the field is evaluated. 

The grid can have different granularities. 

<img src="../images/caldera-map-grid.png" alt="agent-env" width="500" />

This separation is important because it decouples environment semantics (what the world represents) from numerical resolution (how finely it is sampled). As a result, different grid resolutions can represent the same underlying environment, with only the sampling resolution changing. 


In this assignment, we consider three settings of increasing complexity:

* a **deterministic environment** with movement and sampling actions
* a **stochastic environment**, where actions may fail due to the effect of ocean currents
* a **partially observable environment**, where the agent has a limited ability to perceive other vehicles in the environment but must avoid collisions

⚠️Note that across all these settings, the agent may be assigned different missions, such as finding the point of maximal depth or maximizing the informational value of collected samples. These varying objectives are captured through different reward functions, allowing us to study how task definition influences agent behavior.


## General Guidelines

While we provide the entire framework for your implementation east, note that the only files you will submit are:

- implementation.py - where you implement the coding assignment
- answers.py - where you answer the theoretical questions



⚠️ Do not modify the other files - since they will not be examined 

##  🎯 Step 1: Creating the basic environment


As a first step, we will create a basic enviroment, in which the actions are determinsitic and the environment is fully observable. 

The environment is a grid, defined by the dimensions X,Y of the grid, and the cell size, delta. 
The agent is a vehichle of a user-specified size. 


**Actions**

**Observation space**

A valuable observation should contain the information the agent needs to make a decision.

For this mission, the observation needs to include:

- current row position
- current column position
- remaining energy budget
- whether the current cell has already been sampled
- an uncertainty estimate
- local sensor readings
- a map of visited cells
- a compressed belief state over the whole caldera



In [ ]:
import gymnasium as gym
from gymnasium import spaces

example_observation_space = spaces.Dict(
    {
        "position": spaces.Box(low=0, high=4, shape=(2,), dtype=int),
        "energy": spaces.Discrete(21),
        "sampled_here": spaces.Discrete(2),
    }
)

example_action_space = spaces.Discrete(5)

example_observation_space, example_action_space



Our action space is naturally discrete.

We can encode actions as:

- `0`: north
- `1`: south
- `2`: west
- `3`: east
- `4`: sample

This is a good fit for `spaces.Discrete(5)`.


Reward design is where the mission objective becomes explicit.

Adaptive smapling

A simple reward function could be:

- small penalty for movement, to represent energy cost
- larger positive reward for sampling a valuable unsampled cell
- small penalty for sampling a location twice
- episode ends when the energy budget runs out

For adaptive sampling, we want reward to prefer *informative* samples instead of random ones.

One simple approximation is to store a hidden value map:

- ordinary cells give low reward
- scientifically interesting cells give high reward

Then the agent learns that the best policy is not just to move, but to move toward useful places and sample selectively.

In [ ]:
import numpy as np

value_map = np.array(
    [
        [1, 1, 2, 1, 1],
        [1, 2, 4, 2, 1],
        [1, 3, 8, 3, 1],
        [1, 2, 4, 2, 1],
        [1, 1, 2, 1, 1],
    ],
    dtype=float,
)

value_map


## Implementation plan for `CalderaEnv`

Open [implementation.py](C:/Users/sarahk/Documents/PYCHARM-projects/PARLAI/hw/hw1/implementation.py). The class already has the right method names. We just need to define the environment details.

A reasonable implementation plan is:

1. Store mission parameters such as grid size and max energy.
2. Define `observation_space` and `action_space`.
3. In `reset()`, place the robot at a start cell and clear sampled locations.
4. In `step()`, update the state based on the chosen action.
5. Compute a reward and decide whether the episode is over.


In [ ]:
from hw.hw1.implementation import CalderaEnv

print(CalderaEnv)


## What should `reset()` do?

At the start of each episode, we want a fresh mission.

`reset()` should usually:

- restore the robot to a starting location
- reset the remaining energy
- clear the sampled-cell memory
- return the first observation and an `info` dictionary

A typical observation might look like:

```python
{
    "position": np.array([0, 0]),
    "energy": 20,
    "sampled_here": 0,
}
```

## What should `step()` do?

Each step should:

- read the chosen action
- move or sample
- reduce energy
- compute reward
- decide whether the episode is terminated or truncated
- return the next observation and extra info

A clean mental model is:

```python
take action -> update state -> compute reward -> package outputs
```

## Pseudocode

Here is a simple version of the environment logic:

```python
if action is movement:
    move within grid bounds
    reward = -0.1

elif action is sample:
    if current cell was not sampled before:
        reward = scientific_value_of_cell
        mark sampled
    else:
        reward = -0.5

energy -= 1
done = energy == 0
```

## Suggested implementation sketch

The sketch below is not meant to replace your file directly. It is here to show the structure you are aiming for.

In [ ]:
tutorial_code = '''
from typing import Any

import numpy as np
import gymnasium as gym
from gymnasium import spaces


class AGymEnv(gym.Env):
    def __init__(self, a_string: str):
        self.a_string = a_string
        self.grid_size = 5
        self.max_energy = 20
        self.value_map = np.array([
            [1, 1, 2, 1, 1],
            [1, 2, 4, 2, 1],
            [1, 3, 8, 3, 1],
            [1, 2, 4, 2, 1],
            [1, 1, 2, 1, 1],
        ], dtype=float)

        self.observation_space = spaces.Dict({
            "position": spaces.Box(low=0, high=self.grid_size - 1, shape=(2,), dtype=int),
            "energy": spaces.Discrete(self.max_energy + 1),
            "sampled_here": spaces.Discrete(2),
        })

        self.action_space = spaces.Discrete(5)

    def _get_obs(self):
        sampled_here = int(tuple(self.position) in self.sampled_cells)
        return {
            "position": self.position.copy(),
            "energy": self.energy,
            "sampled_here": sampled_here,
        }

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.position = np.array([0, 0], dtype=int)
        self.energy = self.max_energy
        self.sampled_cells = set()
        return self._get_obs(), {}

    def step(self, action: int):
        reward = -0.1
        row, col = self.position

        if action == 0:
            row = max(0, row - 1)
        elif action == 1:
            row = min(self.grid_size - 1, row + 1)
        elif action == 2:
            col = max(0, col - 1)
        elif action == 3:
            col = min(self.grid_size - 1, col + 1)
        elif action == 4:
            cell = tuple(self.position)
            if cell not in self.sampled_cells:
                self.sampled_cells.add(cell)
                reward = float(self.value_map[cell])
            else:
                reward = -0.5

        self.position = np.array([row, col], dtype=int)
        self.energy -= 1

        terminated = self.energy <= 0
        truncated = False
        info = {"mission": self.a_string}

        return self._get_obs(), reward, terminated, truncated, info
'''

print(tutorial_code)


## Why this environment is adaptive

The environment becomes adaptive because the agent gets different value from different sample locations.

If the reward is larger near scientifically interesting parts of the caldera, then the optimal strategy is to:

- travel toward promising areas
- avoid wasting samples on repeated cells
- make good use of a limited energy budget

That is the main idea of adaptive sampling in a compact RL environment.

## Possible extensions

Once the basic version works, you can make the mission more realistic.

Possible extensions include:

- stochastic ocean currents that push the robot
- noisy sensor readings
- a hidden contamination plume
- partial observability
- bonus reward for discovering new high-value regions
- penalties for revisiting too often

These extensions make the environment richer without changing the basic Gymnasium interface.